# ساخت نگاشت موجودیت‌ها (Entities) و روابط (Relations) به فرمت JSON

این نوت‌بوک دو کار انجام می‌دهد:

1. از روی فایل `entity.csv` یک فایل JSON با فرمت `{"m.<freebase_id>": "label, description"}` می‌سازد.
2. از روی فایل `relation.txt` یک فایل JSON با فرمت `{"relation_name": "relation_name"}` می‌سازد (هر رابطه به مقدار خودش نگاشت می‌شود).

**نکته:** مسیر فایل‌های ورودی و خروجی را در سلول تنظیمات زیر مطابق نیاز خودتان تغییر دهید.

In [1]:
import pandas as pd
import json
import os

# ---------------------- تنظیمات مسیرها ----------------------
ENTITY_CSV_PATH   = "entities_data_final.csv"              # مسیر فایل ورودی موجودیت‌ها
RELATION_TXT_PATH = "relations_integrated.txt"            # مسیر فایل ورودی روابط

ENTITY_JSON_OUT   = "entity_text.json"     # خروجی نگاشت موجودیت‌ها
RELATION_JSON_OUT = "relation_text.json"   # خروجی نگاشت روابط


## ۱. ساخت نگاشت موجودیت‌ها از `entity.csv`

In [3]:
# فایل با تب (\t) از هم جدا شده است
df = pd.read_csv(ENTITY_CSV_PATH, sep=",", dtype=str, keep_default_na=False)

print(df.shape)
df.head()


(29177, 9)


,freebase_id,wikidata_id,label,description,has_page,has_data,images,image_count,url
0,0100gqqp,Q16323012,"Grimm, season 4",season of television series,TRUE,TRUE,,0,https://www.wikidata.org/wiki/Q16323012
1,0100nzc6,Q17109012,Lords of the Car Hoards,television series,TRUE,TRUE,,0,https://www.wikidata.org/wiki/Q17109012
2,0100r_fd,Q16961156,Chasing Maria Menounos,television series,TRUE,TRUE,,0,https://www.wikidata.org/wiki/Q16961156
3,01010z9z,NOT_FOUND,Bellator Fighting Championships,Bella Fighting Championships is a TV program.,FALSE,TRUE,,0,https://www.wikidata.org/wiki/NOT_FOUND
4,010154q9,Q16953626,"The Bachelorette, season 10",season of US television series,TRUE,TRUE,,0,https://www.wikidata.org/wiki/Q16953626


In [8]:
# def make_key(freebase_id: str) -> str:
#     """
#     freebase_id در فایل csv به شکل '0100gqqp' است.
#     فرمت استاندارد Freebase mid به صورت 'm.0100gqqp' است،
#     پس در صورتی که پیشوند 'm.' وجود نداشته باشد، اضافه می‌شود.
#     """
#     freebase_id = str(freebase_id).strip()
#     if not freebase_id:
#         return freebase_id
#     if freebase_id.startswith("m.") or freebase_id.startswith("/m/"):
#         return freebase_id.replace("/m/", "m.")
#     return f"{freebase_id}"


def clean_text(text: str) -> str:
    """
    بعضی از متن‌ها در فایل csv به‌خاطر escape شدن کوتیشن‌های داخلی،
    به‌صورت "" (دو کوتیشن پشت‌سرهم) نوشته شده‌اند که باید به یک
    کوتیشن معمولی (") تبدیل شوند. مثال:
        'title character of ""Phineas and Ferb""'
        -> 'title character of "Phineas and Ferb"'
    """
    text = (text or "").strip()
    text = text.replace('""', '"')
    return text


def make_value(label: str, description: str) -> str:
    """
    مقدار value از اتصال label و description ساخته می‌شود.
    اگر هرکدام خالی بودند، آن قسمت حذف می‌شود.
    اگر هر دو خالی بودند، رشته خالی برگردانده می‌شود.
    """
    label = clean_text(label)
    description = clean_text(description)
    
    parts = [p for p in (label, description) if p]
    if not parts:
        return ""
    return ", ".join(parts)


entity_map = {}
for _, row in df.iterrows():
    key = row["freebase_id"]
    if not key:
        continue
    value = make_value(row.get("label", ""), row.get("description", ""))
    entity_map[key] = value

print(f"تعداد موجودیت‌های ساخته‌شده: {len(entity_map)}")
# نمایش چند نمونه
list(entity_map.items())[:5]


تعداد موجودیت‌های ساخته‌شده: 29177


[('0100gqqp', 'Grimm, season 4, season of television series'),
 ('0100nzc6', 'Lords of the Car Hoards, television series'),
 ('0100r_fd', 'Chasing Maria Menounos, television series'),
 ('01010z9z',
  'Bellator Fighting Championships, Bella Fighting Championships is a TV program.'),
 ('010154q9', 'The Bachelorette, season 10, season of US television series')]

In [9]:
with open(ENTITY_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(entity_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {ENTITY_JSON_OUT}")


فایل ذخیره شد: entity_text.json


## ۲. ساخت نگاشت روابط از `relation.txt`

In [6]:
with open(RELATION_TXT_PATH, "r", encoding="utf-8") as f:
    relations = [line.strip() for line in f if line.strip()]

print(f"تعداد روابط: {len(relations)}")
relations[:5]


تعداد روابط: 327


['american_football.football_historical_coach_position',
 'american_football.game_passing_statistics',
 'american_football.game_receiving_statistics',
 'american_football.game_rushing_statistics',
 'american_football.player_game_statistics']

In [7]:
# هر رابطه به مقدار خودش نگاشت می‌شود
relation_map = {rel: rel for rel in relations}

with open(RELATION_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(relation_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {RELATION_JSON_OUT}")
list(relation_map.items())[:5]


فایل ذخیره شد: relation_text.json


[('american_football.football_historical_coach_position',
  'american_football.football_historical_coach_position'),
 ('american_football.game_passing_statistics',
  'american_football.game_passing_statistics'),
 ('american_football.game_receiving_statistics',
  'american_football.game_receiving_statistics'),
 ('american_football.game_rushing_statistics',
  'american_football.game_rushing_statistics'),
 ('american_football.player_game_statistics',
  'american_football.player_game_statistics')]

### توضیح ساختار خروجی‌ها

**entity_id2text.json**
```json
{
  "m.0100gqqp": "Grimm, season 4, season of television series",
  "m.01010z9z": "Bellator Fighting Championships, Bella Fighting Championships is a TV program."
}
```

**relation_id2text.json**
```json
{
  "american_football.player_receiving_statistics": "american_football.player_receiving_statistics",
  "architecture.occupancy": "architecture.occupancy"
}
```
